# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

# <span style="font-size:1.4em;">The ML Pipeline at a Glance</span>

<span style="font-size:1.15em;">

This notebook walks through a complete supervised-learning pipeline for predicting product sales. We load data with `pandas`, split it into train/test sets, fit a baseline with `sklearn`'s `LinearRegression`, engineer richer features with `pandas` and `numpy`, then use Lasso, Ridge, and Elastic Net from `sklearn` to regularize and select the best model via cross-validation.

</span>

# <span style="font-size:1.4em;">Setup & Imports</span>

<span style="font-size:1.15em;">

We import `pandas` for data manipulation, `numpy` for numerical operations, `matplotlib` for plotting, and several modules from `sklearn` (scikit-learn) — the standard Python library for machine learning.

</span>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (LinearRegression, Lasso, Ridge, ElasticNet,
                                  LassoCV, RidgeCV, ElasticNetCV)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# <span style="font-size:1.4em;">Step 1: Load & Explore the Data</span>

<span style="font-size:1.15em;">

We read the H&M sales CSV into a `pandas` DataFrame and preview it. After loading, we filter down to a single product type so the model focuses on one category at a time.

</span>

In [ ]:
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)
print(f"{len(df)} rows, {df.shape[1]} columns")

# Show all columns in the preview
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df.head()

## Filter by Product Type

In [ ]:
PRODUCT_TYPE = 'Vest top'  # Change to any product name (see df['name'].unique())

df = df[df['name'] == PRODUCT_TYPE].copy()
print(f"{PRODUCT_TYPE}: {len(df)} rows, {df['id'].nunique()} products")

# <span style="font-size:1.4em;">Step 2: Train/Test Split</span>

<span style="font-size:1.15em;">

We split by product ID (not by row) so all monthly observations for a given product stay together. 80% of products go to training, 20% are held out for testing. This prevents data leakage.

</span>

In [ ]:
np.random.seed(42)
product_ids = df['id'].unique()
np.random.shuffle(product_ids)
split = int(0.8 * len(product_ids))
train_ids, test_ids = product_ids[:split], product_ids[split:]

print(f"Train: {len(train_ids)} products, Test: {len(test_ids)} products")

---
# <span style="font-size:1.4em;">Part 1: Baseline Linear Regression</span>

<span style="font-size:1.15em;">

We fit a simple OLS regression using just price and month dummies to establish a baseline. The data flows through `StandardScaler` (to normalize features), then into `LinearRegression` from `sklearn`, and we evaluate with `mean_absolute_error`.

</span>

In [ ]:
# Months are already one-hot encoded in the CSV
month_cols = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']

basic_features = ['price'] + month_cols
print(f"Features ({len(basic_features)}): {basic_features}")

In [ ]:
def split_and_scale(data, features, train_ids, test_ids):
    """Split by product ID, extract features, and standardize."""
    tr = data[data['id'].isin(train_ids)]
    te = data[data['id'].isin(test_ids)]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr[features])
    X_te = scaler.transform(te[features])
    return X_tr, X_te, tr['sales'].values, te['sales'].values, scaler

X_train, X_test, y_train, y_test, scaler1 = split_and_scale(df, basic_features, train_ids, test_ids)

# Baseline: plain OLS with just price + month dummies
ols_basic = LinearRegression().fit(X_train, y_train)
mae_basic = mean_absolute_error(y_test, ols_basic.predict(X_test))
print(f"OLS (price + month) — Test MAE: {mae_basic:.1f}")

---
# <span style="font-size:1.4em;">Part 2: Feature Engineering</span>

<span style="font-size:1.15em;">

We use `pandas` and `numpy` to create new features from the raw data — lag sales, rolling averages, price transformations, and interaction terms. More features give the model more signal, but also increase the risk of overfitting, which motivates regularization in Part 3.

</span>

In [ ]:
df_eng = df.copy()

# Lag features: sales from 1, 2, 3 months ago
df_eng['lag_1'] = df_eng.groupby('id')['sales'].shift(1)
df_eng['lag_2'] = df_eng.groupby('id')['sales'].shift(2)
df_eng['lag_3'] = df_eng.groupby('id')['sales'].shift(3)

# Rolling statistics (3-month window, shifted to avoid leakage)
df_eng['ma_3']  = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).mean().shift(1))
df_eng['std_3'] = df_eng.groupby('id')['sales'].transform(lambda x: x.rolling(3).std().shift(1))

# Price features
df_eng['price_pct_change'] = df_eng.groupby('id')['price'].pct_change()
df_eng['price_sq'] = df_eng['price'] ** 2

# Interaction terms
df_eng['price_x_lag_1'] = df_eng['price'] * df_eng['lag_1']
df_eng['lag1_x_lag2']   = df_eng['lag_1'] * df_eng['lag_2']

df_eng.fillna(0, inplace=True)
df_eng.replace([np.inf, -np.inf], 0, inplace=True)

# Color and pattern indicators
color_cols = ['Black', 'Dark Blue', 'White', 'Blue', 'Dark Grey', 'Grey',
              'Light Beige', 'Light Blue', 'Light Pink', 'Beige', 'Dark Red',
              'Greenish Khaki', 'Light Grey', 'Off White', 'Red', 'Pink']
pattern_cols = ['Solid', 'Denim', 'All over pattern', 'Melange', 'Stripe', 'Lace']

# Full feature list
all_features = (['price', 'price_sq', 'price_pct_change',
                 'lag_1', 'lag_2', 'lag_3', 'ma_3', 'std_3',
                 'price_x_lag_1', 'lag1_x_lag2']
                + month_cols + color_cols + pattern_cols)
print(f"{len(all_features)} features: {all_features}")

In [ ]:
X_train2, X_test2, y_train2, y_test2, scaler2 = split_and_scale(df_eng, all_features, train_ids, test_ids)

# OLS with all engineered features — likely overfits
ols_eng = LinearRegression().fit(X_train2, y_train2)
mae_eng_train = mean_absolute_error(y_train2, ols_eng.predict(X_train2))
mae_eng_test  = mean_absolute_error(y_test2, ols_eng.predict(X_test2))

print(f"OLS ({len(all_features)} features)")
print(f"  Train MAE: {mae_eng_train:.1f}")
print(f"  Test  MAE: {mae_eng_test:.1f}")
print(f"  Gap: {mae_eng_test - mae_eng_train:.1f}  (large = overfitting)")

---
# <span style="font-size:1.4em;">Part 3: Regularization</span>

<span style="font-size:1.15em;">

Plain OLS overfits with many features. Here we apply `Lasso`, `Ridge`, and `ElasticNet` from `sklearn` — each adds a penalty controlled by λ that shrinks weights and improves generalization. We sweep a grid of λ values to see how penalty strength trades off between underfitting and overfitting.

</span>

### Lasso: Shrinks + eliminates weights

As λ increases, Lasso pushes more weights to exactly zero — watch the `nonzero` column.

In [ ]:
# Sweep λ values for Lasso — higher λ = more weights pushed to zero
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
lasso_results = []

for lam in lambdas:
    m = Lasso(alpha=lam).fit(X_train2, y_train2)
    lasso_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
        'nonzero':   int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

### Ridge: Shrinks weights but keeps them all

Unlike Lasso, Ridge never zeros out any weight — it shrinks them all proportionally.

In [ ]:
# Sweep λ values for Ridge — shrinks weights but never zeros them out
ridge_results = []

for lam in lambdas:
    m = Ridge(alpha=lam).fit(X_train2, y_train2)
    ridge_results.append({
        'λ': lam,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2)),
    })

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

# <span style="font-size:1.4em;">Model Selection via Cross-Validation</span>

<span style="font-size:1.15em;">

Instead of manually picking λ, we use `LassoCV`, `RidgeCV`, and `ElasticNetCV` from `sklearn` — these try many λ values using cross-validation on the training set and automatically select the one that minimizes error.

</span>

In [ ]:
# Shared penalty grid for all three CV models
lambdas_cv = np.logspace(-3, 3, 50)

# Use cross-validation to select best λ for Lasso and Ridge, best λ + α for Elastic Net
lasso_cv = LassoCV(alphas=lambdas_cv).fit(X_train2, y_train2)
ridge_cv = RidgeCV(alphas=lambdas_cv).fit(X_train2, y_train2)
enet_cv  = ElasticNetCV(alphas=lambdas_cv).fit(X_train2, y_train2)

# Compare all models
print(f"OLS (price + month)  — Test MAE: {mae_basic:.1f}")
print(f"OLS ({len(all_features)} features)    — Test MAE: {mae_eng_test:.1f}")
print(f"Lasso (λ={lasso_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, lasso_cv.predict(X_test2)):.1f}")
print(f"Ridge (λ={ridge_cv.alpha_:.3f})   — Test MAE: {mean_absolute_error(y_test2, ridge_cv.predict(X_test2)):.1f}")
print(f"ElasticNet (λ={enet_cv.alpha_:.3f}, α={enet_cv.l1_ratio_:.2f}) — Test MAE: {mean_absolute_error(y_test2, enet_cv.predict(X_test2)):.1f}")